In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

### Load Data

In [2]:
path = "../../data/raw/PS_20174392719_1491204439457_log.csv"
df = pd.read_csv(path)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


### Feature Selection

In [3]:
def perform_feature_engineering(X):
    df_new = X.copy()
    
    # 1. Hour of Day
    df_new['hour_of_day'] = (df_new['step'] % 24).astype('int8')
    
    # 2. Check Balance Orig -- simpan nilai MENTAH dulu (dipakai flag konsistensi di 2b)
    err_orig_raw = df_new['newbalanceOrig'] + df_new['amount'] - df_new['oldbalanceOrg']
    df_new['errorBalanceOrig'] = err_orig_raw

    # 2b. [PATTERN] Konsistensi saldo pengirim -- WAJIB dari nilai MENTAH (sebelum signed-log).
    # Threshold 1.0 untuk mengabaikan noise floating-point.
    df_new['orig_balance_consistency'] = np.where(
        np.abs(err_orig_raw) > 1.0, 'Orig_Mismatch', 'Orig_Consistent'
    )
    
    # 3. Check Balance Dest & Flagging Merchant
    df_new['is_merchant_dest'] = df_new['nameDest'].str.startswith('M').astype('int8')
    df_new['errorBalanceDest'] = np.where(
        df_new['is_merchant_dest'] == 1, 0,
        df_new['oldbalanceDest'] + df_new['amount'] - df_new['newbalanceDest']
    )

    # 3b. [PATTERN] Jenis pihak tujuan (merchant vs customer)
    df_new['dest_kind'] = np.where(df_new['is_merchant_dest'] == 1, 'Dest_Merchant', 'Dest_Customer')

    df_new['errorBalanceDest'] = (
        np.sign(df_new['errorBalanceDest']) * np.log1p(np.abs(df_new['errorBalanceDest']))
    )
    df_new['errorBalanceOrig'] = (
        np.sign(df_new['errorBalanceOrig']) * np.log1p(np.abs(df_new['errorBalanceOrig']))
    )

    # 4 Balance Drain Ratio
    df_new['has_zero_orig_balance'] = (df_new['oldbalanceOrg'] == 0).astype('int8')
    df_new['balance_drain_ratio'] = np.where(
        df_new['oldbalanceOrg'] > 0,
        df_new['amount'] / df_new['oldbalanceOrg'],
        0
    )
    df_new['balance_drain_ratio'] = df_new['balance_drain_ratio'].clip(0, 10)

    # 4b. [PATTERN] Kategori pengurasan saldo -- skala 0..1 (proporsi saldo yang terambil),
    # dihitung dari nilai MENTAH sebelum log1p di step 6b.
    drain01 = np.clip(
        np.where(df_new['oldbalanceOrg'] > 0, df_new['amount'] / df_new['oldbalanceOrg'], 0), 0, 1
    )
    df_new['drain_category'] = pd.cut(
        drain01, bins=[-0.01, 0.25, 0.90, 1.01],
        labels=['Low_Drain', 'Mid_Drain', 'Full_Drain']
    )

    # 4c. [PATTERN] Apakah rekening pengirim dikosongkan total
    df_new['emptied_origin'] = np.where(
        (df_new['oldbalanceOrg'] > 0) & (df_new['newbalanceOrig'] == 0), 'Emptied', 'Not_Emptied'
    )

    # 5. Binning Time Segmentation
    bins_time = [-1, 6, 18, 24]
    labels_time = ['Midnight_to_Morning', 'Working_Hours', 'Evening']
    df_new['time_segment'] = pd.cut(df_new['hour_of_day'], bins=bins_time, labels=labels_time)

    # 6. Binning Amount (qcut) -- dihitung dari amount MENTAH (sebelum log di step 6b)
    labels_amount = ['Low_Amount', 'Medium_Amount', 'High_Amount']
    df_new['amount_category'] = pd.qcut(df_new['amount'], q=3, labels=labels_amount)

    # 6b. Log-transform fitur moneter berekor sangat panjang (amount & saldo)
    # Justifikasi: amount, oldbalanceOrg, oldbalanceDest sangat right-skewed
    # (max puluhan juta). Tanpa log1p, scaler apa pun menyisakan ekor ekstrem yang
    # mendominasi jarak Euclidean sehingga clustering praktis hanya "melihat" fitur ini.
    # log1p memampatkan skala tanpa mengubah urutan. Kolom ini non-negatif -> aman.
    # Semua fitur turunan (errorBalance*, balance_drain_ratio, kategori PATTERN) sudah
    # dihitung dari nilai MENTAH di atas, jadi transformasi ini tidak merusak maknanya.
    for col in ['amount', 'oldbalanceOrg', 'oldbalanceDest']:
        df_new[col] = np.log1p(df_new[col])

    # 7. Drop noise & redundan feature
    cols_to_drop = [
        'nameOrig', 'nameDest', 'newbalanceDest', 'newbalanceOrig',
        'step', 'is_merchant_dest', 'isFraud', 'isFlaggedFraud'
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df_new.columns]
    df_new = df_new.drop(columns=cols_to_drop)
    
    return df_new


Justification:
1. Temporal Extraction
The default step feature is linear sequential. Through a modulo operation (step % 24), the data is transformed into a daily temporal cycle representation (0–23). The next binning step into three business segments (Midnight_to_Morning, Working_Hours, Evening) is based on empirical EDA findings showing extreme fraud rate spikes (>20%) in the early morning (off-peak hours). This grouping helps the model distinguish normal human operational patterns from structured attacks.

2. Check Financial Logic
In the financial system, the ending balance must be mathematically balanced against the beginning balance minus/plus the transaction amount ($Saldo_{akhir} = Saldo_{awal} \pm Amount$). The creation of an error balance feature on the sender and receiver sides aims to explicitly capture this logical deviation. A value of $\neq 0$ in this feature is a strong anomaly (hard signal) indicating account manipulation, system bypass, or indication of money laundering (layering).

3. Handling Merchant Limitations
The dataset has a limitation where all transactions with a Merchant destination do not track balance changes, so by design the destination balance value is set to 0. If calculated directly, this would create a large false anomaly in the errorBalanceDest. The is_merchant_dest flag is used as a filter to force the merchant error balance value to 0, so that the clustering algorithm is not distorted by this data recording error.

4. Data Transformation Using Logarithms
The distribution of error balance values ​​is severely right-skewed and bimodal due to the large number of 0 values ​​(merchant effect). If directly included in the distance weighting (scaling), these extreme outliers would dominate the calculation and distort the inter-quartile range (IQR). Using the Signed Log Transform ($sign(x) \times \ln(|x| + 1)$) is the absolute solution for drastically reducing extreme skewness, while maintaining the original transaction direction/sign (+/-) without producing undefined/error values ​​at 0.

5. Account Drainage Ratio
Fraudsters have a tendency to completely drain victims' accounts (complete account draining). The balance_drain_ratio feature captures the intensity of this draining. The has_zero_orig_balance feature is then used to set a ratio of 0 for accounts with no initial balance. Furthermore, a limit on extreme values ​​(.clip(0, 10)) is applied to control the variance caused by the division of very small floating-point values.

6. Amount Category Discretization
Some association algorithms or rules (such as Apriori) require purely categorical input data. Transforming the numeric amounts into three tertile levels using pd.qcut ensures a perfectly balanced distribution of the data frequencies (33.33% each). This maximizes the entropy value of feature information and prevents algorithm bias toward certain nominal classes.

7. Eliminating redundancy and data leakage
nameOrig & nameDest are discarded because their high cardinality provides no analytical value but only increases computational burden. newbalanceOrig & newbalanceDest are discarded because their information has already been extracted and is more informatively represented by the error balance feature (avoiding perfect multicollinearity). isFraud & isFlaggedFraud must be discarded because they are target labels that must not leak (data leakage) into the unsupervised learning/data mining process.

In [4]:
# Kontrak hand-off Data Engineer -> Pattern Analyst.
# 7 atribut perilaku KATEGORIKAL ini adalah SATU-SATUNYA sumber untuk Phase 3
# (association rule mining DAN anomaly pattern validation), sehingga tidak ada
# lagi duplikasi rekayasa fitur di sisi Pattern Analyst.
PATTERN_COLS = [
    'type',                       # jenis transaksi
    'amount_category',            # tertil nominal (Low/Medium/High)
    'time_segment',               # segmen waktu harian
    'drain_category',             # proporsi saldo terkuras (Low/Mid/Full)
    'emptied_origin',             # rekening pengirim dikosongkan total?
    'orig_balance_consistency',   # saldo pengirim konsisten secara logika?
    'dest_kind',                  # tujuan merchant atau customer?
]

def select_segmentation_features(X):
    # Clustering hanya memakai fitur numerik + 'type' (di-OHE oleh ColumnTransformer).
    # Seluruh kolom kategorikal turunan lain dibuang agar tidak ikut ke pipeline numerik.
    drop_cols = [c for c in PATTERN_COLS if c != 'type']
    return X.drop(columns=drop_cols, errors='ignore')

def select_pattern_features(X):
    return X[PATTERN_COLS].astype('object')

### Pipeline preprocessing

In [5]:
nums_cols = ['amount', 'oldbalanceOrg', 'oldbalanceDest', 'errorBalanceOrig', 'errorBalanceDest', 'balance_drain_ratio', 'hour_of_day']
cat_cols = ['type']
binary_cols = ['has_zero_orig_balance']

# Numeric pipeline: StandardScaler menyetarakan variance tiap fitur ke ~1
# (membagi dengan std SELURUH distribusi, jadi tahan terhadap zero-inflation --
# tidak seperti RobustScaler yang meledakkan errorBalanceDest karena IQR~0 akibat 65% nol),
# lalu clip [-5, 5] untuk menjinakkan sisa ekor ekstrem. Tujuannya: tidak ada satu
# fitur pun yang mendominasi jarak Euclidean, sehingga clustering representatif.
def clip_range(A):
    return np.clip(A, -5, 5)

num_pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('clip', FunctionTransformer(clip_range))
])

preprocessor_seg = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, nums_cols),
        ('cat', OneHotEncoder(drop=None, sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('bin', 'passthrough', binary_cols)
    ],
    remainder='drop'
)

segmentation_pipeline = Pipeline([
    ('core_eng', FunctionTransformer(perform_feature_engineering)),
    ('selector', FunctionTransformer(select_segmentation_features)),
    ('preprocessing', preprocessor_seg)
])

pattern_pipeline = Pipeline([
    ('core_eng', FunctionTransformer(perform_feature_engineering)),
    ('selector', FunctionTransformer(select_pattern_features))
])

In [6]:
data_seg_array = segmentation_pipeline.fit_transform(df)

In [7]:
# 1. Clustering
feature_names = (
    nums_cols + 
    list(segmentation_pipeline.named_steps['preprocessing'].transformers_[1][1].get_feature_names_out(cat_cols)) +
    binary_cols
)
df_clustering_final = pd.DataFrame(data_seg_array, columns=feature_names)
df_clustering_final.to_parquet('../../data/processed/real_used/data_phase2_clustering.parquet', index=False)
print("shape clustering: ", df_clustering_final.shape)

# 2. Apriori
df_rules_final = pattern_pipeline.fit_transform(df)
df_rules_final.to_parquet('../../data/processed/real_used/data_phase3_rules.parquet', index=False)
print("shape apriori: ", df_rules_final.shape)

shape clustering:  (6362620, 13)
shape apriori:  (6362620, 7)


In [9]:
df_clustering_final.head()

,amount,oldbalanceOrg,oldbalanceDest,errorBalanceOrig,errorBalanceDest,balance_drain_ratio,hour_of_day,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,has_zero_orig_balance
0,-0.907462,0.816576,-1.144463,-1.81572,-0.420546,-0.572945,-3.313772,0.0,0.0,0.0,1.0,0.0,0.0
1,-1.824023,0.449673,-1.144463,-1.81572,-0.420546,-0.564644,-3.313772,0.0,0.0,0.0,1.0,0.0,0.0
2,-3.106552,-0.389888,-1.144463,-1.81572,0.477755,-0.311374,-3.313772,0.0,0.0,0.0,0.0,1.0,0.0
3,-3.106552,-0.389888,0.331751,-1.81572,1.300354,-0.311374,-3.313772,0.0,1.0,0.0,0.0,0.0,0.0
4,-0.813537,0.567961,-1.144463,-1.81572,-0.420546,-0.511045,-3.313772,0.0,0.0,0.0,1.0,0.0,0.0


In [10]:
df_rules_final.head()

,type,amount_category,time_segment,drain_category,emptied_origin,orig_balance_consistency,dest_kind
0,PAYMENT,Low_Amount,Midnight_to_Morning,Low_Drain,Not_Emptied,Orig_Consistent,Dest_Merchant
1,PAYMENT,Low_Amount,Midnight_to_Morning,Low_Drain,Not_Emptied,Orig_Consistent,Dest_Merchant
2,TRANSFER,Low_Amount,Midnight_to_Morning,Full_Drain,Emptied,Orig_Consistent,Dest_Customer
3,CASH_OUT,Low_Amount,Midnight_to_Morning,Full_Drain,Emptied,Orig_Consistent,Dest_Customer
4,PAYMENT,Low_Amount,Midnight_to_Morning,Mid_Drain,Not_Emptied,Orig_Consistent,Dest_Merchant


In [11]:
df_clustering_final.var().values

array([0.99961038, 1.00000016, 1.00000016, 1.00000016, 1.00000016,
       1.00000016, 1.00000016, 0.17155668, 0.22799626, 0.00646938,
       0.22380334, 0.07674113, 0.22124863])

In [12]:
df_clustering_final.describe()

,amount,oldbalanceOrg,oldbalanceDest,errorBalanceOrig,errorBalanceDest,balance_drain_ratio,hour_of_day,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,has_zero_orig_balance
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,3.691983e-05,-9.119791e-17,-1.012039e-16,6.175156e-17,-1.829676e-17,6.861284e-18,1.326515e-16,2.199226e-01,3.516633e-01,6.511783e-03,3.381461e-01,8.375622e-02,3.304376e-01
std,9.998052e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,4.141940e-01,4.774895e-01,8.043246e-02,4.730786e-01,2.770219e-01,4.703707e-01
min,-5.000000e+00,-1.307741e+00,-1.144463e+00,-1.817743e+00,-3.552638e+00,-5.890014e-01,-3.545157e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,-7.377024e-01,-1.307741e+00,-1.144463e+00,-1.910344e-01,-4.205459e-01,-5.890014e-01,-7.685352e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.108966e-01,3.786861e-01,6.036893e-01,4.485370e-01,-4.205459e-01,-5.676766e-01,1.570054e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,7.759050e-01,7.352982e-01,8.943044e-01,7.109228e-01,-4.188283e-01,2.717727e-02,8.511608e-01,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00
max,4.134041e+00,1.849876e+00,1.773668e+00,1.913345e+00,2.409522e+00,2.187269e+00,1.776701e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [13]:
# === VERIFIKASI SKALA: pastikan tidak ada 1 fitur yang mendominasi jarak Euclidean ===
from sklearn.decomposition import PCA

_v = df_clustering_final.var().sort_values(ascending=False)
print("Variance per fitur (urut desc):")
print(_v.round(3).to_string())

_num_min = df_clustering_final[nums_cols].var().min()
print(f"\nRasio variance numerik terbesar/terkecil: {_v[nums_cols].max()/_num_min:.1f}  (target: << 100)")

# PCA: kalau PC1 masih ~0.99 berarti masih ada fitur yang mendominasi
_idx = np.random.RandomState(42).choice(len(df_clustering_final), 200_000, replace=False)
_pca = PCA(n_components=5).fit(df_clustering_final.values[_idx])
print("\nPCA explained_variance_ratio:", np.round(_pca.explained_variance_ratio_, 4))
_cols = list(df_clustering_final.columns)
_top = np.argsort(np.abs(_pca.components_[0]))[::-1][:3]
print("PC1 loading teratas:", [(_cols[i], round(float(_pca.components_[0][i]), 3)) for i in _top])
print("\n-> BERHASIL jika PC1 sudah TIDAK lagi ~0.99 dan loading-nya menyebar ke banyak fitur.")

Variance per fitur (urut desc):
balance_drain_ratio      1.000
errorBalanceDest         1.000
oldbalanceDest           1.000
errorBalanceOrig         1.000
oldbalanceOrg            1.000
hour_of_day              1.000
amount                   1.000
type_CASH_OUT            0.228
type_PAYMENT             0.224
has_zero_orig_balance    0.221
type_CASH_IN             0.172
type_TRANSFER            0.077
type_DEBIT               0.006

Rasio variance numerik terbesar/terkecil: 1.0  (target: << 100)

PCA explained_variance_ratio: [0.342  0.2126 0.1375 0.1245 0.0876]
PC1 loading teratas: [('amount', 0.529), ('errorBalanceOrig', 0.515), ('oldbalanceDest', 0.476)]

-> BERHASIL jika PC1 sudah TIDAK lagi ~0.99 dan loading-nya menyebar ke banyak fitur.
